In [0]:
df = spark.read.table('workspace.pyspark_learning.countries_consolidated')
df.display()

## SQL string as expression

In [0]:
# imoort expr
from pyspark.sql.functions import expr

In [0]:
"""
using withColumn to create an additionl column for 'population_class', using a SQL case stsatement
NOTE:  note the use of 'case when' for first statment and 'when' only for additional statements
NOTE:  'end' to finialize the statement
"""

df.withColumn('population_class', expr("case when population > 300000000 then 'High' when population > 30000000 then 'Medium' else 'Small' end")).display()

In [0]:
"""
It can be cumbersome to write the expression above as a single string therefore you can put your SQL in a varaible, multiline using python's tripple-quote
"""

sql_string = """
case 
    when population > 300000000 then 'High' 
    when population > 30000000 then 'Medium' 
    else 'Small' 
end
"""

In [0]:
"""
re-execute the withColumn operation but use the variable
"""

df.withColumn('population_class', expr(sql_string)).display()

In [0]:
"""
The code below is equivalent to above but uses the Select method and not withColunm.
Here, instead of adding the new column to the df, one must select all the columns they wish to see in the output
Last, one must create an alias to name the new column.
"""

df.select("country", "region", "population", expr(sql_string).alias("population_class")).display()
    


- use 'withColumn' when you want to see the whole DF and a new column
- use 'select' with 'alias' when you want to see only specific columns, else 'withColumn' is more efficient

In [0]:
"""
the code below combines select and expr using 'selectExpr', which is part of the dataframe API
NOTE: uses the SQL string from the variable above
NOTE: the new column is named after the expression, not great. 'alias' does not work
"""

df.selectExpr("country", "population", sql_string).display()

In [0]:
"""
Add the alias to selectExpr in order to name the column
NOTE: 'end as' then column name 'population_class'
"""

df.selectExpr("country", "region", "population", "case when population > 300000000 then 'High' when population > 30000000 then 'Medium' else 'Small' end as population_class").display()


## Condition NOT using SQL syntax

In [0]:
"""
for non-sql syntax, import 'when' and chain them
NOTE:  need column objects thus using dot-notation
NOTE:  note the use of 'otherwise' for the else condtion
"""
from pyspark.sql.functions import when

df.withColumn(
    'population_class', 
    when(df.population > 300000000, 'High'). \
    when(df.population > 30000000, 'Medium').\
    otherwise('Small')). \
    display()

In [0]:

"""
this is the same as the code above but uses bracket notation to get the column objects
"""

df.withColumn(
    'population_class', 
    when(df['population']> 300000000, 'High'). \
    when(df['population'] > 30000000, 'Medium').\
    otherwise('Small')). \
    display()

In [0]:
"""
Below is chaining 'when', with Select
NOTE: all selected columns, including the 'when' and 'otherwise' functions is INSIDE the 'select' function
"""
df.select('country', 'population', \
    when(df.population > 30000000, 'High'). \
    when(df.population > 3000000, 'Medium'). \
    otherwise('Small'). \
    alias('population_class')). \
    display()

## Conditional Functions
https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#conditional-functions

These function include:
- when()
- coalesce()

In [0]:
"""
Coalesce example

"""
data = [
    (1, "Alice", None, 23),
    (2, None, "Ally", 25),
    (4, None, None, 31),
    (3, "Bob", "Bobby", 38),

]
schema = ['id', 'name', 'nickname', 'age']

df2 = spark.createDataFrame(data, schema=schema)
df2.display()

In [0]:
from pyspark.sql.functions import coalesce

In [0]:
"""
Coalesce:  Returns the first column value which is not NULL
The code below creates a column 'name_found', with name or nickname so we do not get any null values
Will display name or nickname, first one it finds, in the new column
"""

df2.select('id', 'name', 'nickname', 'age', coalesce('name','nickname').alias('name_found')).display()